# [16.5] VLM Modality and Region SHAP Exercises

Metadata: `EXERCISE_ID=16_5_vlm_modality_and_region_shap`, `GT_TIER=GT-0`, `EXPECTED_RUNTIME=seconds for toy contracts; minutes for CUDA report`, `REQUIRES_GPU=True`.

In [ ]:
from __future__ import annotations

import itertools
import math
import sys
from collections.abc import Callable, Mapping
from dataclasses import dataclass
from pathlib import Path

import torch as t

chapter = "chapter16_shapley_attribution_baselines"
section = "part5_vlm_modality_region_shap"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part5_vlm_modality_region_shap.tests as tests
import part5_vlm_modality_region_shap.utils as utils

Coalition = frozenset[int]


In [ ]:
@dataclass(frozen=True)
class ShapleyEfficiencyReport:
    shapley_sum: float
    total_value_delta: float
    efficiency_error: float
    satisfies_efficiency: bool


@dataclass(frozen=True)
class VLMModalitySHAPReport:
    modality_values: t.Tensor
    baseline_score: float
    image_only_score: float
    text_only_score: float
    full_score: float
    synergy: float
    detects_synergy: bool
    satisfies_efficiency: bool


@dataclass(frozen=True)
class VLMRegionSHAPReport:
    region_values: t.Tensor
    region_names: tuple[str, ...]
    target_region: str
    target_value: float
    max_background_value: float
    localizes_target: bool
    satisfies_efficiency: bool


## Exact Shapley Values

Implement exact weighted marginal contributions and the efficiency report.

In [ ]:
def all_coalitions(num_players: int) -> tuple[Coalition, ...]:
    raise NotImplementedError()


def normalize_coalition_values(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
) -> dict[Coalition, float]:
    raise NotImplementedError()


def exact_shapley_values(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
) -> t.Tensor:
    raise NotImplementedError()


def shapley_efficiency_report(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
    tolerance: float = 1e-9,
) -> ShapleyEfficiencyReport:
    raise NotImplementedError()


## Modality SHAP

Implement the image/text game and detect synergy.

In [ ]:
def coalition_values_from_function(
    num_players: int,
    value_fn: Callable[[Coalition], float],
) -> dict[Coalition, float]:
    raise NotImplementedError()


def vlm_modality_game(
    *,
    image_weight: float = 1.0,
    text_weight: float = 0.5,
    synergy_weight: float = 2.0,
) -> dict[Coalition, float]:
    raise NotImplementedError()


def vlm_modality_shap_report(
    *,
    image_weight: float = 1.0,
    text_weight: float = 0.5,
    synergy_weight: float = 2.0,
    min_synergy: float = 1.0,
    tolerance: float = 1e-9,
) -> VLMModalitySHAPReport:
    raise NotImplementedError()


## Region SHAP

Implement the object/background/OCR game and target-localization report.

In [ ]:
def vlm_region_game(
    *,
    object_weight: float = 2.0,
    ocr_weight: float = 0.75,
    object_ocr_interaction: float = 0.5,
) -> dict[Coalition, float]:
    raise NotImplementedError()


def vlm_region_shap_report(
    *,
    region_names: tuple[str, ...] = ("object", "background", "ocr_text"),
    target_region: str = "object",
    min_margin: float = 0.5,
    tolerance: float = 1e-9,
) -> VLMRegionSHAPReport:
    raise NotImplementedError()


## Notebook Contract

Expose JSON-serializable smoke tests for the deterministic toy contract.

In [ ]:
def _tensor_report(report) -> dict:
    result = report.__dict__.copy()
    for key, value in list(result.items()):
        if hasattr(value, "tolist"):
            result[key] = value.tolist()
    return result


def modality_shap_smoke_test() -> dict:
    return _tensor_report(vlm_modality_shap_report())


def region_shap_smoke_test() -> dict:
    return _tensor_report(vlm_region_shap_report())


def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    return {
        "modality": modality_shap_smoke_test(),
        "region": region_shap_smoke_test(),
    }


## Visible Tests

In [ ]:
tests.test_exact_shapley_values_splits_two_player_synergy(exact_shapley_values)
tests.test_shapley_efficiency_report_requires_complete_coalition_table(
    shapley_efficiency_report
)
tests.test_vlm_modality_game_contains_expected_image_text_coalitions(vlm_modality_game)
tests.test_vlm_modality_shap_report_detects_synergy_and_efficiency(
    vlm_modality_shap_report
)
tests.test_vlm_region_game_keeps_background_as_negative_control(vlm_region_game)
tests.test_vlm_region_shap_report_localizes_object_region(vlm_region_shap_report)
tests.test_modality_shap_smoke_test(modality_shap_smoke_test)
tests.test_region_shap_smoke_test(region_shap_smoke_test)
tests.test_notebook_contract(run_smoke_test)


## Full Verification Contract

The smoke tests check the local exercise implementation. This final cell checks the committed CUDA verification report for the section-scale run and exposes the same `run_gpu_test` / `run_full_experiment` surface used by the release gate.


In [ ]:
def _load_committed_gpu_report() -> dict:
    import json

    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = _load_committed_gpu_report()
{key: gpu[key] for key in [
    "device",
    "preflight_passed",
    "peak_vram_gb",
] if key in gpu}
